In [1]:
import torch
from torch import nn

In [2]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [3]:
common_dir = "../../Common/VAENet"
model.load_state_dict(torch.load(f"{common_dir}/pre_trained_w_encoder.pt", weights_only=True))

<All keys matched successfully>

In [4]:
tensor_x = torch.rand((1, 3, 128, 256), dtype=torch.float32)
torch.onnx.export(
    model,
    (tensor_x,),
    f"{model.__class__.__name__}.onnx",  # filename of the ONNX model
    input_names=["input"],  # Rename inputs for the ONNX model
    output_names=["output"],  # Rename outputs for the ONNX model
    dynamo=False,  # True or False to select the exporter to use
)

In [5]:
!../onnx2c/build/onnx2c vaemodel1.onnx > C/vaemodel1.c

In [ ]:
import onnx
import numpy as np
import os

# Load the ONNX model
onnx_model = onnx.load("vaemodel1.onnx")

# Create directory for weights if it doesn't exist
os.makedirs("C/weights", exist_ok=True)

# Dictionary to store tensors
weight_dict = {}

# Extract all initializers (weights and biases)
for initializer in onnx_model.graph.initializer:
    weight_dict[initializer.name] = onnx.numpy_helper.to_array(initializer)

# Extract and save the specific tensors
if "tensor_onnx__Conv_71" in weight_dict:
    np.save("C/weights/tensor_onnx__Conv_71.npy", weight_dict["tensor_onnx__Conv_71"])
    print(f"Saved tensor_onnx__Conv_71 with shape {weight_dict['tensor_onnx__Conv_71'].shape}")

if "tensor_onnx__Conv_74" in weight_dict:
    np.save("C/weights/tensor_onnx__Conv_74.npy", weight_dict["tensor_onnx__Conv_74"])
    print(f"Saved tensor_onnx__Conv_74 with shape {weight_dict['tensor_onnx__Conv_74'].shape}")

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split

img_path = f"{common_dir}/dataset/"


# Dataset with Transformation
dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
              mean=[0.0, 0.0, 0.0], 
              std=[1.0, 1.0, 1.0])])
)



# Data split
trainpctg  = 0.8
train_size = int(trainpctg * len(dataset))
test_size = len(dataset) - train_size
trainset, testset = random_split(dataset, [train_size, test_size])

print(f"Training set size: {len(trainset)}, Testing set size: {len(testset)}")

Training set size: 34432, Testing set size: 8608


In [7]:
import torch

# Get the first image and its label from the dataset
image, label = dataset[0]

# Convert the image tensor to a byte tensor
image_bytes = torch.ByteTensor(image.mul(255).byte())

# Save the byte tensor to a binary file
with open('C/test_image.bin', 'wb') as f:
    f.write(image_bytes.numpy().tobytes())

print("First image exported to 'test_image.bin'")

First image exported to 'test_image.bin'
